# Challenge — Module 2.4: CSV → SQLite ETL

**Format:** one ETL pipeline. You must use every concept from 2.4: raw `sqlite3` with parameterized queries, bulk insert via `executemany`, `pandas.read_sql`, `DataFrame.to_sql`, and proper transaction handling (`commit`). The PostgreSQL section (2.4 ex_67–68) is theoretical here — write a short comment explaining what would change if the target were Postgres + `psycopg2`. Plus everything from 2.1–2.3.

**Scenario.** You're building a small ETL job: read a CSV of customer transactions, load it into a SQLite database in two staging tables, then run an aggregation query through pandas and write the result back as a third table.

## The problem

Setup writes:

- `transactions.csv`: `txn_id, customer_id, amount, txn_ts`
- `customers.csv`: `customer_id, name, country`

### Your task

1. **Create** a fresh SQLite DB at `etl.db` (delete it if it already exists, for idempotency).
2. **Create two tables** via raw `sqlite3`:
   - `customers(customer_id TEXT PRIMARY KEY, name TEXT, country TEXT)`
   - `transactions(txn_id TEXT PRIMARY KEY, customer_id TEXT, amount REAL, txn_ts TEXT)`
3. **Bulk-insert** `customers.csv` rows using `cursor.executemany(...)` and a parameterized `INSERT`. `commit()` afterwards.
4. **Load** `transactions.csv` into the `transactions` table via `pandas.DataFrame.to_sql(..., if_exists='append', index=False)`.
5. **Query** with `pandas.read_sql`:
   ```sql
   SELECT c.country,
          COUNT(*)      AS txn_count,
          SUM(t.amount) AS total_amount
   FROM transactions t
   JOIN customers c ON c.customer_id = t.customer_id
   GROUP BY c.country
   ORDER BY total_amount DESC;
   ```
6. **Write** the resulting DataFrame back to a new table `country_summary` via `to_sql(..., if_exists='replace')`.
7. **Verify** with a raw `cursor.execute("SELECT * FROM country_summary")` + `fetchall()` and print the rows.

Also: use a **parameterized** `SELECT` (with `?` placeholder, not f-string) to fetch and print the total amount for one specific `customer_id` of your choice.

### Expected output (shape)

```
country_summary:
  ('US', 4, 410.5)
  ('BR', 3, 285.0)
  ('DE', 2, 95.0)

customer C001 total = 175.50
```

(Numbers will match the setup data.)

### Restrictions

- **NEVER** use an f-string or `%`-formatting to build SQL with user values. Always parameterize.
- Wrap connection usage in `with sqlite3.connect(...) as conn:` so transactions close cleanly.
- The PostgreSQL comment (at least 2 lines) must mention: (a) `psycopg2.connect` with a connection string and (b) that `executemany` is slow on Postgres — `COPY FROM STDIN` is the bulk-loading pattern.

### Hints

- `cursor.executemany('INSERT INTO customers VALUES (?, ?, ?)', list_of_tuples)`.
- `pd.read_sql(sql, conn)` returns a DataFrame directly.
- After `to_sql(..., if_exists='replace')`, the table schema is inferred from dtypes — that's fine here.

In [ ]:
# --- setup: writes the two CSVs ---
import csv, os

customers = [
    ['customer_id', 'name',   'country'],
    ['C001',        'Alice',  'US'],
    ['C002',        'Bruno',  'BR'],
    ['C003',        'Clara',  'DE'],
    ['C004',        'Diego',  'BR'],
    ['C005',        'Emma',   'US'],
]
transactions = [
    ['txn_id', 'customer_id', 'amount', 'txn_ts'],
    ['T001', 'C001', '100.00', '2026-04-01T10:00:00'],
    ['T002', 'C001', '75.50',  '2026-04-03T12:30:00'],
    ['T003', 'C002', '120.00', '2026-04-05T09:15:00'],
    ['T004', 'C003', '45.00',  '2026-04-06T16:45:00'],
    ['T005', 'C004', '90.00',  '2026-04-07T11:00:00'],
    ['T006', 'C005', '60.00',  '2026-04-09T14:20:00'],
    ['T007', 'C005', '175.00', '2026-04-12T08:00:00'],
    ['T008', 'C002', '75.00',  '2026-04-15T19:00:00'],
    ['T009', 'C003', '50.00',  '2026-04-18T13:30:00'],
]
with open('customers.csv', 'w', newline='', encoding='utf-8') as f:
    csv.writer(f).writerows(customers)
with open('transactions.csv', 'w', newline='', encoding='utf-8') as f:
    csv.writer(f).writerows(transactions)

# clean slate for the DB
if os.path.exists('etl.db'):
    os.remove('etl.db')
print('setup done')

setup done


In [ ]:
# your code here
import sqlite3
import csv
import pandas as pd

# 1. connect, CREATE TABLE customers, CREATE TABLE transactions
# 2. read customers.csv, executemany INSERT
# 3. read transactions.csv with pandas, .to_sql(append)
# 4. pd.read_sql(JOIN + GROUP BY) -> summary df
# 5. summary.to_sql('country_summary', conn, if_exists='replace', index=False)
# 6. cursor.execute('SELECT * FROM country_summary').fetchall(), print
# 7. parameterized SELECT for one customer total

# --- Postgres notes ---
# If the target were Postgres instead of SQLite:
# - ...
# - ...
